# Predicting Film Success Using Screenplay Structure
### Rick Lee, Michael Xu, Allen Xu

### Imports

In [ ]:
import pandas as pd
import time
import os
import requests
import re
from dotenv import load_dotenv
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

### List of all 1,076 Film Screenplays

In [7]:
movie_titles = [
    "10 Things I Hate About You", "12", "12 and Holding", "12 Monkeys", "12 Years a Slave",
    "127 Hours", "1492: Conquest of Paradise", "15 Minutes", "17 Again", "187",
    "2001: A Space Odyssey", "2012", "25th Hour", "30 Minutes or Less", "42",
    "44 Inch Chest", "48 Hrs.", "50/50", "500 Days of Summer", "8mm", "9",
    "A Few Good Men", "A Serious Man", "Above the Law", "Absolute Power", "The Abyss",
    "Ace Ventura: Pet Detective", "Adaptation", "The Addams Family", "The Adjustment Bureau",
    "The Adventures of Buckaroo Banzai Across the Eighth Dimension", "Affliction",
    "After School Special", "After.Life", "Agnes of God", "Air Force One", "Airplane",
    "Airplane 2: The Sequel", "Aladdin", "Ali", "Alien 3", "Alien Nation", "Alien vs. Predator",
    "Alien: Resurrection", "Aliens", "All About Eve", "All About Steve", "All the King's Men",
    "All the President's Men", "Almost Famous", "Alone in the Dark", "Amadeus", "Amelia",
    "American Beauty", "American Gangster", "American Graffiti", "American History X",
    "American Hustle", "American Madness", "American Outlaws", "American Pie",
    "The American President", "American Psycho", "American Shaolin: King of Kickboxers II",
    "American Splendor", "An American Werewolf in London", "The American", "The Amityville Asylum",
    "Amour", "An Education", "Analyze That", "Analyze This", "Anastasia", "Angel Eyes",
    "Angels & Demons", "Anna Karenina", "Annie Hall", "The Anniversary Party", "Anonymous",
    "Antitrust", "Antz", "The Apartment", "Apocalypse Now", "April Fool's Day", "Apt Pupil",
    "Arbitrage", "Arcade", "Arctic Blue", "Argo", "Armageddon", "Army of Darkness",
    "Arsenic and Old Lace", "Arthur", "The Artist", "As Good As It Gets", "Assassins",
    "The Assignment", "At First Sight", "August: Osage County", "Austin Powers - International Man of Mystery",
    "Austin Powers - The Spy Who Shagged Me", "Authors Anonymous", "Autumn in New York", "Avatar",
    "The Avengers", "The Avengers", "L'Avventura", "Awakenings",
    "Babel", "Bachelor Party", "The Bachelor Party", "The Back-up Plan", "Backdraft", "Bad Boys",
    "Bad Country", "Bad Day at Black Rock", "Bad Dreams", "Bad Lieutenant", "Bad Santa",
    "Bad Teacher", "Badlands", "Bamboozled", "Barry Lyndon", "Barton Fink", "Basic",
    "Basic Instinct", "Basquiat", "Batman", "Batman Returns", "The Battle of Algiers",
    "The Battle of Shaker Heights", "Battle: Los Angeles", "The Beach", "Bean",
    "Beasts of the Southern Wild", "Beavis and Butt-head Do America", "Beginners", "Being Human",
    "Being John Malkovich", "Being There", "The Believer", "Beloved", "Benny & Joon",
    "The Best Exotic Marigold Hotel", "Big", "The Big Blue", "Big Fish", "The Big Lebowski",
    "The Big White", "The Birds", "Birthday Girl", "The Black Dahlia", "Black Rain",
    "Black Snake Moan", "Black Swan", "Blade", "Blade II", "Blade Runner", "Blade: Trinity",
    "The Blast from the Past", "The Blind Side", "The Bling Ring", "Blood and Wine",
    "Blood Simple", "Blow", "Blue Valentine", "Blue Velvet", "Bodies, Rest & Motion", "Body Heat",
    "Body of Evidence", "The Bodyguard", "Bones", "The Bonfire of the Vanities", "Bonnie and Clyde",
    "Boogie Nights", "The Book of Eli", "Boondock Saints 2: All Saints Day", "The Boondock Saints",
    "Bottle Rocket", "Bound", "The Bounty Hunter", "The Bourne Identity", "The Bourne Supremacy",
    "The Bourne Ultimatum", "The Box", "Braveheart", "Brazil", "Break", "Breakdown",
    "The Breakfast Club", "Breaking Away", "Brick", "Bridesmaids", "Bringing Out the Dead",
    "Broadcast News", "Broken Arrow", "Broken Embraces", "The Brothers Bloom", "Bruce Almighty",
    "Buffy the Vampire Slayer", "Bull Durham", "Buried", "Burlesque", "Burn After Reading",
    "Burning Annie", "The Butterfly Effect", "The Cable Guy", "Candle to Water", "Capote", "Carrie",
    "Cars 2", "Case 39", "Casino", "Cast Away", "Catch Me If You Can", "Catwoman", "Cecil B. Demented",
    "Cedar Rapids", "Celeste & Jesse Forever", "The Cell", "Cellular", "The Change-Up",
    "Changeling", "Chaos", "Charade", "Charlie's Angels", "Chasing Amy", "Chasing Sleep",
    "Cherry Falls", "Chinatown", "Christ Complex", "Chronicle",
    "The Chronicles of Narnia: The Lion, the Witch and the Wardrobe", "The Cider House Rules",
    "The Cincinnati Kid", "Cinema Paradiso", "Cirque du Freak: The Vampire's Assistant",
    "Citizen Kane", "City of Joy", "Clash of the Titans", "Clerks", "Cliffhanger", "Clueless", "Cobb",
    "Code of Silence", "Cold Mountain", "Collateral", "Collateral Damage", "Colombiana",
    "Color of Night", "Commando", "Conan the Barbarian", "Confessions of a Dangerous Mind",
    "Confidence", "Constantine", "The Cooler", "Copycat", "Coraline", "Coriolanus",
    "Cowboys & Aliens", "Cradle 2 the Grave", "Crank", "Crash", "Crazy, Stupid, Love", "Crazylove",
    "Creation", "Crime Spree", "The Croods", "Crouching Tiger, Hidden Dragon", "Croupier",
    "The Crow Salvation", "The Crow", "The Crow: City of Angels", "Cruel Intentions", "The Crying Game",
    "Cube", "The Curious Case of Benjamin Button", "Custody", "The Damned United",
    "Dances with Wolves", "Dark City", "The Dark Knight Rises", "Dark Star", "Darkman", "Date Night",
    "Dave Barry's Complete Guide to Guys", "Dawn of the Dead", "Day of the Dead",
    "The Day the Clown Cried", "The Day the Earth Stood Still", "Days of Heaven",
    "Dead Poets Society", "Death at a Funeral", "Death to Smoochy", "The Debt", "Deception",
    "Deep Cover", "Deep Rising", "The Deer Hunter", "Defiance", "The Departed", "The Descendants",
    "Despicable Me 2", "Detroit Rock City", "Devil in a Blue Dress", "The Devil's Advocate", "Die Hard",
    "Die Hard 2", "Diner", "The Distinguished Gentleman", "Disturbia", "Django Unchained",
    "Do The Right Thing", "Dog Day Afternoon", "Dogma", "Donnie Brasco", "The Doors",
    "Double Indemnity", "Drag Me to Hell", "Dragonslayer", "Drive", "Drive Angry",
    "Drop Dead Gorgeous", "A Dry White Season", "Duck Soup", "Dumb and Dumber", "Dune", "E.T. the Extra-Terrestrial",
    "Eagle Eye", "Eastern Promises", "Easy A", "Ed TV", "Ed Wood", "Edward Scissorhands",
    "Eight Legged Freaks", "El Mariachi", "Election", "The Elephant Man", "Elizabeth: The Golden Age",
    "Enemy of the State", "The English Patient", "Enough", "Entrapment", "Erik the Viking",
    "Erin Brockovich", "Escape From L.A.", "Escape From New York",
    "Eternal Sunshine of the Spotless Mind", "Even Cowgirls Get the Blues", "Event Horizon",
    "Evil Dead", "Evil Dead II", "Excalibur", "eXistenZ", "Extract",
    "The Fabulous Baker Boys", "Face/Off", "Fair Game", "The Family Man", "Fantastic Four",
    "Fantastic Mr. Fox", "Fargo", "Fast Times at Ridgemont High", "Fatal Instinct",
    "The Fault in Our Stars", "Fear and Loathing in Las Vegas", "Feast", "Ferris Bueller's Day Off",
    "Field of Dreams", "The Fifth Element", "Fight Club", "The Fighter", "Final Destination",
    "Final Destination 2", "Finding Nemo", "Five Easy Pieces", "Flash Gordon", "Fletch", "Flight",
    "The Flintstones", "Forrest Gump", "The Four Feathers", "Four Rooms", "Foxcatcher", "Fracture",
    "Frances", "Frankenstein", "Freaked", "Freddy vs. Jason", "The French Connection", "Frequency",
    "Friday the 13th", "Friday the 13th Part VIII: Jason Takes Manhattan", "Fright Night",
    "Fright Night", "From Dusk Till Dawn", "From Here to Eternity", "Frozen",
    "Frozen", "Frozen River", "Fruitvale Station", "The Fugitive", "Funny People",
    "G.I. Jane", "G.I. Joe: The Rise of Cobra", "Game 6", "The Game", "Gamer", "Gandhi",
    "Gang Related", "Gangs of New York", "Garden State", "Gattaca", "Get Carter", "Get Low",
    "Get Shorty", "The Getaway", "Ghost", "The Ghost and the Darkness", "Ghost Rider",
    "Ghost Ship", "Ghost World", "Ghostbusters", "Ghostbusters II", "Ginger Snaps",
    "The Girl with the Dragon Tattoo", "Gladiator", "Glengarry Glen Ross", "Go", "The Godfather",
    "The Godfather Part II", "The Godfather Part III", "Gods and Monsters", "Godzilla",
    "Gone in 60 Seconds", "The Good Girl", "Good Will Hunting", "Gothika", "The Graduate",
    "Gran Torino", "Grand Hotel", "Grand Theft Parsons", "The Grapes of Wrath", "Gravity",
    "The Green Mile", "Gremlins", "Gremlins 2: The New Batch", "The Grifters", "Grosse Pointe Blank",
    "Groundhog Day", "The Grudge", "Hackers", "Hall Pass", "Halloween: The Curse of Michael Myers",
    "Hancock", "The Hangover", "Hanna", "Hannah and Her Sisters", "Hannibal",
    "Happy Birthday, Wanda June", "Happy Feet", "Hard Rain", "Hard to Kill",
    "Harold & Kumar Go to White Castle", "The Haunting", "He's Just Not That Into You", "Heat",
    "Heathers", "Heavenly Creatures", "Heavy Metal", "The Hebrew Hammer", "Heist",
    "Hellbound: Hellraiser II", "Hellboy", "Hellboy II: The Golden Army", "Hellraiser",
    "Hellraiser III: Hell on Earth", "Hellraiser: Deader", "Hellraiser: Hellseeker", "The Help",
    "Henry Fool", "Henry's Crime", "Hesher", "High Fidelity", "Highlander", "Highlander: Endgame",
    "The Hills Have Eyes", "His Girl Friday", "Hitchcock", "The Hitchhiker's Guide to the Galaxy",
    "Hollow Man", "Honeydripper", "Horrible Bosses", "The Horse Whisperer", "The Hospital",
    "Hostage", "Hot Tub Time Machine", "Hotel Rwanda", "House of 1000 Corpses",
    "How to Lose Friends & Alienate People", "How to Train Your Dragon", "How to Train Your Dragon 2",
    "Hudson Hawk", "The Hudsucker Proxy", "Human Nature", "The Hunt for Red October",
    "I Am Number Four", "I Am Sam", "I Love You Phillip Morris", "I Spit on Your Grave",
    "I Still Know What You Did Last Summer", "I'll Do Anything", "I, Robot", "The Ice Storm",
    "The Ides of March", "The Imaginarium of Doctor Parnassus", "In the Bedroom", "In the Loop",
    "Inception", "The Incredibles", "Independence Day", "Indiana Jones and the Last Crusade",
    "Raiders of the Lost Ark", "Indiana Jones and the Temple of Doom",
    "Indiana Jones and the Kingdom of the Crystal Skull", "The Informant!", "Inglourious Basterds", "The Insider", "Insidious",
    "Insomnia", "Interstellar", "Interview with the Vampire", "Into the Wild", "Intolerable Cruelty",
    "Inventing the Abbotts", "The Invention of Lying", "Invictus", "The Iron Lady", "The Island",
    "It Happened One Night", "It's a Wonderful Life", "It's Complicated", "The Italian Job",
    "The Jacket", "Jackie Brown", "Jacob's Ladder", "Jane Eyre", "Jason X", "Jaws", "Jaws 2",
    "Jay and Silent Bob Strike Back", "Jennifer 8", "Jennifer's Body", "Jerry Maguire", "JFK",
    "Jimmy and Judy", "John Q", "Judge Dredd", "Juno", "Jurassic Park", "Jurassic Park III",
    "The Lost World: Jurassic Park", "Kafka", "Kalifornia", "Kate & Leopold", "Kids",
    "The Kids Are All Right", "Kill Bill", "Killing Zoe", "King Kong",
    "The King of Comedy", "The King's Speech", "The Kingdom", "Klute", "Knocked Up",
    "Kramer vs. Kramer", "Kundun", "Kung Fu Panda", "L.A. Confidential", "Labor of Love", "Labyrinth",
    "The Ladykillers", "Lake Placid", "Land of the Dead", "Larry Crowne", "The Last Boy Scout",
    "Last Chance Harvey", "The Last Flight", "The Last of the Mohicans", "The Last Samurai",
    "The Last Station", "Last Tango in Paris", "Law Abiding Citizen", "Leaving Las Vegas",
    "Legally Blonde", "Legend", "Legion", "Les Misérables", "Leviathan", "Liar Liar", "Life",
    "Life as a House", "The Life of David Gale", "Life of Pi", "Light Sleeper", "The Limey",
    "Limitless", "Lincoln", "The Lincoln Lawyer", "Little Athens", "The Little Mermaid",
    "Little Nicky", "Living in Oblivion", "Lock, Stock and Two Smoking Barrels", "Logan's Run",
    "Lone Star", "The Long Kiss Goodnight", "Looper", "Lord of Illusions",
    "The Lord of the Rings: The Fellowship of the Ring", "The Lord of the Rings: The Return of the King",
    "The Lord of the Rings: The Two Towers", "Lord of War", "The Losers", "Lost Highway",
    "Lost Horizon", "Lost in Space", "Lost in Translation", "Love & Basketball", "Machete",
    "Machine Gun Preacher", "Mad Max 2", "Made", "Magnolia",
    "The Majestic", "Major League", "Malcolm X", "Malibu's Most Wanted",
    "The Man in the Iron Mask", "Man on Fire", "Man on the Moon", "Man Trouble",
    "The Man Who Knew Too Much", "The Man Who Wasn't There", "The Manchurian Candidate",
    "Manhattan Murder Mystery", "Manhunter", "Margaret", "Margin Call", "Margot at the Wedding",
    "El Mariachi", "Marley & Me", "Martha Marcy May Marlene", "Marty", "Mary Poppins", "The Mask",
    "Master and Commander: The Far Side of the World", "The Master", "The Matrix Reloaded", "The Matrix", "Max Payne",
    "Mean Streets", "The Mechanic", "Meet Joe Black", "Meet John Doe", "Megamind", "Memento",
    "Men in Black", "Men in Black 3", "The Men Who Stare at Goats", "Metro", "Miami Vice",
    "Midnight Cowboy", "Midnight Express", "Midnight in Paris",
    "Mighty Morphin Power Rangers: The Movie", "Milk", "Miller's Crossing", "Mimic",
    "Mini's First Time", "Minority Report", "The Miracle Worker", "Mirrors", "Misery",
    "Mission: Impossible", "Mission: Impossible II", "Mission to Mars", "Moneyball", "Monkeybone",
    "Monte Carlo", "Moon", "Moonrise Kingdom", "Moonstruck", "Mr. Blandings Builds His Dream House",
    "Mr. Brooks", "Mr. Deeds Goes to Town", "Mrs. Brown", "Mud", "Mulan", "Mulholland Drive",
    "Mumford", "The Mummy", "Music of the Heart", "Mute Witness", "My Best Friend's Wedding",
    "My Girl", "My Mother Dreams the Satan's Disciples in New York", "My Week with Marilyn",
    "Mystery Men", "Nashville", "Natural Born Killers", "Never Been Kissed", "The NeverEnding Story",
    "New York Minute", "Newsies", "Next", "Next Friday", "The Next Three Days", "Nick of Time",
    "Night Time", "Nightbreed", "The Nightmare Before Christmas",
    "A Nightmare on Elm Street", "A Nightmare on Elm Street: The Dream Child", "Nine", "The Nines",
    "Ninja Assassin", "Ninotchka", "The Ninth Gate", "No Strings Attached", "Notting Hill",
    "Nurse Betty", "O Brother, Where Art Thou?", "Oblivion", "Observe and Report", "Obsessed",
    "Ocean's Eleven", "Ocean's Twelve", "Office Space", "The Omega Man", "One Flew Over the Cuckoo's Nest",
    "Only God Forgives", "Ordinary People", "Orgy of the Dead", "Orphan", "The Other Boleyn Girl",
    "Out of Sight", "The Pacifier", "Pandorum", "Panic Room", "Papadopoulos & Sons", "ParaNorman",
    "Pariah", "The Passion of Joan of Arc", "The Patriot", "Paul", "Pearl Harbor", "Peeping Tom",
    "Peggy Sue Got Married", "Perfect Creature", "A Perfect World", "The Perks of Being a Wallflower",
    "Pet Sematary", "Pet Sematary II", "Petulia", "Philadelphia", "Phone Booth", "Pi", "The Pianist",
    "The Piano", "Pineapple Express", "Pirates of the Caribbean: The Curse of the Black Pearl",
    "Pirates of the Caribbean: Dead Man's Chest", "Pitch Black", "Planet of the Apes",
    "Platinum Blonde", "Platoon", "Pleasantville", "Point Break", "Pokémon: Mewtwo Returns",
    "The Postman", "The Power of One", "Precious", "Predator", "Pretty Woman",
    "Pride & Prejudice", "Priest", "The Princess Bride", "The Private Life of Sherlock Holmes",
    "The Producers", "The Program", "Prom Night", "Prometheus", "The Prophecy", "The Proposal",
    "Psycho", "Public Enemies", "Pulp Fiction", "Punch-Drunk Love", "Purple Rain", "Quantum Project",
    "Queen of the Damned", "The Queen", "Rachel Getting Married", "Raging Bull", "Raising Arizona",
    "Rambling Rose", "Rambo: First Blood Part II", "The Reader", "Real Genius",
    "Rear Window", "Rebel Without a Cause", "Red Planet", "Red Riding Hood", "Reindeer Games",
    "The Relic", "Remember Me", "The Replacements", "Repo Man", "The Rescuers Down Under",
    "Reservoir Dogs", "Resident Evil", "Return of the Apes", "The Revenant", "Revolutionary Road",
    "Ringu", "Rise of the Guardians", "Rise of the Planet of the Apes", "RKO 281", "The Road",
    "Robin Hood: Prince of Thieves", "The Rock", "RocknRolla", "Rocky",
    "The Rocky Horror Picture Show", "Romeo + Juliet", "Ronin", "The Roommate", "Roughshod",
    "The Ruins", "Runaway Bride", "Rush", "Rush Hour", "Rush Hour 2", "Rushmore", "Rust and Bone",
    "S. Darko", "The Saint", "The Salton Sea", "The Sandlot", "Save the Last Dance",
    "Saving Mr. Banks", "Saving Private Ryan", "Saw", "Scarface", "Schindler's List",
    "Scott Pilgrim vs. the World", "Scream", "Scream 2", "Scream 3", "Se7en", "The Searchers",
    "Semi-Pro", "Sense and Sensibility", "Serenity", "Serial Mom", "The Sessions",
    "The Seventh Seal", "Sex and the City", "Sex, Lies, and Videotape", "Sexual Life",
    "Shakespeare in Love", "Shallow Grave", "Shame", "Shampoo", "The Shawshank Redemption",
    "She's Out of My League", "Sherlock Holmes", "Shifty", "The Shining", "The Shipping News",
    "Shivers", "Shrek", "Shrek the Third", "Sideways", "The Siege", "Signs", "The Silence of the Lambs",
    "Silver Bullet", "Silver Linings Playbook", "S1m0ne", "Single White Female", "Sister Act",
    "Six Degrees of Separation", "The Sixth Sense", "Sleepless in Seattle", "Sleepy Hollow",
    "Sling Blade", "Slither", "Slumdog Millionaire", "Smashed", "Smokin' Aces", "Snatch",
    "Snow Falling on Cedars", "Snow White and the Huntsman", "So I Married an Axe Murderer",
    "The Social Network", "Solaris", "Soldier", "Someone to Watch Over Me", "Something's Gotta Give",
    "Source Code", "South Park: Bigger, Longer & Uncut", "Spanglish", "Spare Me", "Spartan", "Speed Racer", "Sphere",
    "Spider-Man", "St. Elmo's Fire", "Star Trek", "Star Trek II: The Wrath of Khan",
    "Star Trek: First Contact", "Star Trek: Generations", "Star Trek: Nemesis",
    "Star Trek: The Motion Picture", "Star Wars: Episode IV - A New Hope", "Star Wars: Episode II - Attack of the Clones",
    "Star Wars: Episode VI - Return of the Jedi", "Star Wars: Episode III - Revenge of the Sith",
    "Star Wars: Episode V - The Empire Strikes Back", "Star Wars: Episode VII - The Force Awakens",
    "Star Wars: Episode I - The Phantom Menace", "Starman", "Starship Troopers", "State and Main",
    "Station West", "Stepmom", "The Sting", "Stir of Echoes", "Storytelling", "Strange Days",
    "Strangers on a Train", "The Stunt Man", "Sugar", "Sugar & Spice", "Sunset Blvd.",
    "Sunshine Cleaning", "Super 8", "Superbad", "Supergirl", "The Surfer King", "Surrogates",
    "Suspect Zero", "Sweeney Todd: The Demon Barber of Fleet Street", "The Sweet Hereafter",
    "Sweet Smell of Success", "Swingers", "Swordfish", "Synecdoche, New York", "Syriana",
    "Take Shelter", "Taking Lives", "The Taking of Pelham One Two Three", "Taking Sides",
    "The Talented Mr. Ripley", "Tall in the Saddle", "Tamara Drewe", "Taxi Driver", "Ted",
    "The Terminator", "Terminator 2: Judgment Day", "Terminator Salvation", "The Rage: Carrie 2",
    "Thelma & Louise", "There's Something About Mary", "They", "The Thing",
    "The Things My Father Never Taught Me", "Thirteen Days", "This Boy's Life", "This Is 40",
    "Thor", "Three Kings", "Three Kings", "Three Men and a Baby",
    "The Three Musketeers", "Thunderbirds", "Thunderheart", "Ticker", "Timber Falls",
    "The Time Machine", "Tin Cup", "Tin Men", "Tinker Tailor Soldier Spy", "Titanic", "TMNT",
    "To Sleep with Anger", "Tombstone", "Tomorrow Never Dies", "Top Gun", "Total Recall",
    "The Tourist", "Toy Story", "Traffic", "Training Day", "Trainspotting",
    "Transformers: The Movie", "Tremors", "Tristan & Isolde", "TRON", "TRON: Legacy",
    "Tropic Thunder", "True Grit", "True Lies", "True Romance", "The Truman Show", "Twilight",
    "The Twilight Saga: New Moon", "Twin Peaks", "Twins", "Two for the Money", "U Turn", "The Ugly Truth",
    "Unbreakable", "Under Fire", "Unknown", "Up", "Up in the Air", "The Usual Suspects",
    "V for Vendetta", "Valkyrie", "Vanilla Sky", "The Verdict", "Very Bad Things", "The Village",
    "Virtuosity", "The Visitor", "Wag the Dog", "A Walk to Remember", "Walking Tall", "Wall Street",
    "Wall Street: Money Never Sleeps", "WALL-E", "Wanted", "War Horse", "War of the Worlds",
    "Warm Springs", "Warrior", "Watchmen", "Water for Elephants", "The Way Back", "We Own the Night",
    "What About Bob?", "What Lies Beneath", "When a Stranger Calls", "While She Was Out",
    "The Whistleblower", "White Christmas", "White Jazz", "The White Ribbon", "White Squall",
    "Whiteout", "Who Framed Roger Rabbit", "Who's Your Daddy?", "Wild at Heart", "The Wild Bunch",
    "Wild Hogs", "Wild Things", "Wild Things: Diamonds in the Rough", "Wild Wild West", "Willow",
    "Win Win", "Wind Chill", "Withnail & I", "Witness", "The Wizard of Oz",
    "The Wolf of Wall Street", "Wonder Boys", "The Woodsman", "The World Is Not Enough",
    "The Wrestler", "The X Files", "X-Men", "X-Men Origins: Wolverine", "xXx",
    "Year One", "Yes Man", "You Can Count on Me", "You've Got Mail", "Youth in Revolt",
    "Zero Dark Thirty", "Zerophilia"
]

### Obtain IMDb Ratings via API Call and Create CSV File
#### You should only run this ONCE! It will cache movie ratings in movie_ratings_final.csv
#### Might take a while

### Preprocess Screenplay Text, Obtain Features, and use Logistic Regression (~0.52 Accuracy)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction import text
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
import numpy as np

def clean_screenplay(text):
    if not text:
        return ""

    text = re.sub(r'(INT\.|EXT\.)\s+.*', '', text)

    text = re.sub(r'^[A-Z\s\d]+$', '', text, flags=re.MULTILINE)

    text = re.sub(r'\(.*?\)', '', text)

    text = re.sub(r'[A-Z ]+:', '', text)

    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df = pd.read_csv('movie_ratings_final.csv')

def to_filesystem_name(title):
    articles = ("The ", "A ", "An ", "L'", "El ", "La ", "Le ")
    for a in articles:
        if title.startswith(a):
            if a == "L'":
                return f"{title[2:]}, L'"
            return f"{title[len(a):]}, {a.strip()}"
    return title

def load_script_text(clean_title):
    fs_title = to_filesystem_name(clean_title)
    file_path = f"./screenplay_dataset/Script_{fs_title}.txt"

    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            raw_text = f.read()
            word_count = len(raw_text.split())
            scenes = len(re.findall(r'INT\.|EXT\.', raw_text.upper()))

            lines = raw_text.split('\n')
            avg_line_length = np.mean([len(l.split()) for l in lines if l.strip()] if lines else 0)
            exclamation_count = raw_text.count('!') / (word_count + 1)
            cleaned_text = clean_screenplay(raw_text)

            return pd.Series([cleaned_text, word_count, scenes, avg_line_length, exclamation_count])
    return pd.Series([None, 0, 0, 0, 0])

print("Loading script texts...")
print(df['original_title'])
df[['script_text', 'word_count', 'scenes', 'avg_line_length', 'excl_count']] = df['original_title'].apply(load_script_text)

df = df.dropna(subset=['script_text'])

screenplay_stop_words = ['written', 'screenplay', 'draft', 'final', 'revised', 'shooting', 'script', 'page', 'contd', 'continued', 'based', 'novel', 'story', 'david', 'jr', '2007', '11', 'continued', 'vo', 'os', 'ext', 'int', 'scene', 'day', 'night', '97', '10', '04']
final_stop_words = text.ENGLISH_STOP_WORDS.union(screenplay_stop_words)

tfidf = TfidfVectorizer(stop_words=list(final_stop_words), max_features=10000, ngram_range=(1,2), min_df=5, max_df=0.8)

X = tfidf.fit_transform(df['script_text'])
y = df['is_successful']

print("Extracting structural features...")
struct_features = df[['word_count', 'scenes', 'avg_line_length', 'excl_count']].values

scaler = StandardScaler()
struct_features_scaled = scaler.fit_transform(struct_features)

X_comb = hstack([X, struct_features_scaled])

X_train, X_test, y_train, y_test = train_test_split(X_comb, y, test_size=0.2, random_state=67)

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n--- Model Performance ---")
print(classification_report(y_test, y_pred))

Loading script texts...
0       10 Things I Hate About You
1                               12
2                   12 and Holding
3                       12 Monkeys
4                 12 Years a Slave
                   ...            
1071           You Can Count on Me
1072               You've Got Mail
1073               Youth in Revolt
1074              Zero Dark Thirty
1075                    Zerophilia
Name: original_title, Length: 1076, dtype: str
Extracting structural features...

--- Model Performance ---
              precision    recall  f1-score   support

           0       0.41      0.46      0.43        85
           1       0.53      0.48      0.50       108

    accuracy                           0.47       193
   macro avg       0.47      0.47      0.47       193
weighted avg       0.48      0.47      0.47       193



### Classification with Random Forest (also poor accuracy)

In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X_comb, y, test_size=0.2, random_state=67)

log_reg = LogisticRegression(class_weight='balanced', max_iter=1000)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=67)

log_reg.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

print("="*30)
print(f"{'MODEL':<20} | {'ACCURACY':<10}")
print("-"*30)
print(f"{'Logistic Regression':<20} | {accuracy_score(y_test, y_pred_log):.4f}")
print(f"{'Random Forest':<20} | {accuracy_score(y_test, y_pred_rf):.4f}")
print("="*30)

print("\nDetailed Random Forest Report:")
print(classification_report(y_test, y_pred_rf))

MODEL                | ACCURACY  
------------------------------
Logistic Regression  | 0.4715
Random Forest        | 0.4974

Detailed Random Forest Report:
              precision    recall  f1-score   support

           0       0.44      0.48      0.46        85
           1       0.56      0.51      0.53       108

    accuracy                           0.50       193
   macro avg       0.50      0.50      0.49       193
weighted avg       0.50      0.50      0.50       193



### Classification with XGBoost

In [ ]:
from xgboost import XGBClassifier

num_neg = len(y[y == 0])
num_pos = len(y[y == 1])
ratio = num_neg / num_pos

# XGBoost Model
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=ratio,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

print(f"XGBoost Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\n--- XGBoost Classification Report ---")
print(classification_report(y_test, y_pred))

c:\Users\litte\miniconda3\envs\cs3630\Lib\site-packages\xgboost\training.py:200: UserWarning: [01:49:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.4663

--- XGBoost Classification Report ---
              precision    recall  f1-score   support

           0       0.42      0.59      0.49        85
           1       0.53      0.37      0.44       108

    accuracy                           0.47       193
   macro avg       0.48      0.48      0.46       193
weighted avg       0.49      0.47      0.46       193



### Regression on IMDb Scores, Evaluated Using Margins (e.g., within 0.5 of actual rating)
#### Achieved very low R^2 value (~0.04) despite solid accuracy
#### So, we decided to pivot from predicting commercial success (IMDb rating) to analyzing screenplay quality

In [37]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

y = df['rating']
X_train, X_test, y_train, y_test = train_test_split(X_comb, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, max_depth=50, min_samples_leaf=2, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# this is my marginal/tolerance accuracy! it will calculate the percentage of predictions
# that fall within a distance "margin" from the actual rating.
def calculate_tolerance_accuracy(y_true, y_pred, margin=0.75):
    diff = np.abs(y_true - y_pred)
    correct_within_margin = np.sum(diff <= margin)
    return (correct_within_margin / len(y_true)) * 100

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
ballpark_acc = calculate_tolerance_accuracy(y_test, y_pred)

print("--- Professional Performance Summary ---")
print(f"Mean Absolute Error: {mae:.2f} stars")
print(f"R² Score (Variance Explained): {r2:.2f}")
print("-" * 40)
print(f"PRO-METRIC: The model predicts the rating within 0.75 point")
print(f"of the actual score {ballpark_acc:.1f}% of the time.")
print("-" * 40)

--- Professional Performance Summary ---
Mean Absolute Error: 0.73 stars
R² Score (Variance Explained): 0.00
----------------------------------------
PRO-METRIC: The model predicts the rating within 0.75 point
of the actual score 61.7% of the time.
----------------------------------------


### Determined most important features (e.g., line length, exclamation density)
#### However, we determined that it would be better to use more complex features (e.g., dialogue-to-action ratio, pacing, etc.)

In [38]:
import matplotlib.pyplot as plt

tfidf_features = list(tfidf.get_feature_names_out())

struct_names = ['Word Count', 'Scene Count', 'Avg Line Length', 'Exclamation Density']
all_feature_names = tfidf_features + struct_names

importances = model.feature_importances_

feature_importance_dict = dict(zip(all_feature_names, importances))
sorted_importance = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)

print("--- Top 15 Most Important Features ---")
for name, score in sorted_importance[:15]:
    print(f"{name:<20}: {score:.4f}")

--- Top 15 Most Important Features ---
Avg Line Length     : 0.0930
Exclamation Density : 0.0731
Word Count          : 0.0525
Scene Count         : 0.0525
love                : 0.0204
second              : 0.0136
edgar               : 0.0127
black               : 0.0094
october             : 0.0079
august              : 0.0069
81                  : 0.0068
2009                : 0.0068
original            : 0.0067
paul                : 0.0065
dead                : 0.0062


### Performed a Dimensionality Reduction Experiment to see if reducing dimensionality would increase model accuracy
#### It did, but only to a limited extent

In [ ]:
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

print("="*60)
print("DIMENSIONALITY REDUCTION EXPERIMENT")
print("="*60)

# Test different dimensionality levels
component_tests = [50, 100, 200]
results_summary = []

for n_components in component_tests:
    print(f"\n--- Testing with {n_components} SVD components ---")

    # Apply SVD to TF-IDF matrix only (keep structural features separate)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_tfidf_reduced = svd.fit_transform(X)

    # Combine reduced TF-IDF (as sparse matrix) with structural features
    X_tfidf_sparse = csr_matrix(X_tfidf_reduced)
    X_reduced_comb = hstack([X_tfidf_sparse, struct_features_scaled])

    # Test 1: Classification
    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
        X_reduced_comb, df['is_successful'], test_size=0.2, random_state=67
    )

    rf_reduced = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=67)
    rf_reduced.fit(X_train_c, y_train_c)
    y_pred_c = rf_reduced.predict(X_test_c)
    acc_reduced = accuracy_score(y_test_c, y_pred_c)
    print(f"  Classification Accuracy: {acc_reduced:.4f}")

    # Test 2: Regression (rating prediction)
    X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
        X_reduced_comb, df['rating'], test_size=0.2, random_state=42
    )

    rf_reg_reduced = RandomForestRegressor(n_estimators=200, max_depth=30, random_state=42)
    rf_reg_reduced.fit(X_train_r, y_train_r)
    y_pred_r = rf_reg_reduced.predict(X_test_r)

    r2_reduced = r2_score(y_test_r, y_pred_r)
    mae_reduced = mean_absolute_error(y_test_r, y_pred_r)
    tolerance_reduced = calculate_tolerance_accuracy(y_test_r, y_pred_r)

    print(f"  Regression R²: {r2_reduced:.4f} | MAE: {mae_reduced:.2f} | Tolerance: {tolerance_reduced:.1f}%")

    results_summary.append({
        'Components': n_components,
        'Clf Accuracy': f"{acc_reduced:.4f}",
        'Reg R²': f"{r2_reduced:.4f}",
        'Reg MAE': f"{mae_reduced:.2f}",
        'Tolerance %': f"{tolerance_reduced:.1f}"
    })

print("\n" + "="*60)
print("COMPARISON: ORIGINAL vs REDUCED DIMENSIONALITY")
print("="*60)
print(f"Original (10000 features):")
print(f"  Classification: 0.5000 (approx)")
print(f"  Regression R²: {r2:.4f} | MAE: {mae:.2f} | Tolerance: {ballpark_acc:.1f}%")

print("\nReduced Dimensionality Results:")
for result in results_summary:
    print(f"\n{result['Components']} components:")
    print(f"  Classification Acc: {result['Clf Accuracy']}")
    print(f"  Regression R²: {result['Reg R²']} | MAE: {result['Reg MAE']} | Tolerance: {result['Tolerance %']}%")


DIMENSIONALITY REDUCTION EXPERIMENT

--- Testing with 50 SVD components ---
  Classification Accuracy: 0.5130
  Regression R²: 0.0363 | MAE: 0.72 | Tolerance: 64.2%

--- Testing with 100 SVD components ---
  Classification Accuracy: 0.4922
  Regression R²: 0.0151 | MAE: 0.74 | Tolerance: 60.1%

--- Testing with 200 SVD components ---
  Classification Accuracy: 0.5181
  Regression R²: -0.0039 | MAE: 0.73 | Tolerance: 61.1%

COMPARISON: ORIGINAL vs REDUCED DIMENSIONALITY
Original (10000 features):
  Classification: 0.5000 (approx)
  Regression R²: 0.0020 | MAE: 0.73 | Tolerance: 61.7%

Reduced Dimensionality Results:

50 components:
  Classification Acc: 0.5130
  Regression R²: 0.0363 | MAE: 0.72 | Tolerance: 64.2%

100 components:
  Classification Acc: 0.4922
  Regression R²: 0.0151 | MAE: 0.74 | Tolerance: 60.1%

200 components:
  Classification Acc: 0.5181
  Regression R²: -0.0039 | MAE: 0.73 | Tolerance: 61.1%


### (TO BE REVIEWED) We pivoted to analyzing screenplay structure and have the following pipeline

In [ ]:
print("="*70)
print("SCREENPLAY STRUCTURE ANALYSIS - NEW APPROACH")
print("="*70)
print("\nExtracting screenplay craft metrics from raw text...\n")

df_raw = pd.read_csv('movie_ratings_final.csv')
df_raw = df_raw[df_raw['rating'].notna()].reset_index(drop=True)

def extract_screenplay_metrics(screenplay_text, raw_text):
    if not raw_text or len(raw_text) < 100:
        return pd.Series([0]*15)

    # 1. DIALOGUE vs ACTION RATIO
    # Count parentheticals (action descriptions) vs dialogue
    action_lines = len(re.findall(r'^\s{0,10}[A-Z][A-Z\s\(\)\.,-]*$', raw_text, re.MULTILINE))
    dialogue_lines = len(re.findall(r'^\s{20,}[A-Z].*$', raw_text, re.MULTILINE))
    dialogue_ratio = dialogue_lines / (action_lines + dialogue_lines + 1) if (action_lines + dialogue_lines) > 0 else 0

    # 2. SCENE COMPLEXITY: Measure unique character names per scene
    # Look for character introductions (ALL CAPS on their own line followed by dialogue)
    character_pattern = r'^([A-Z][A-Z\s]*?)\n\s{0,10}[A-Za-z]'
    character_matches = re.findall(character_pattern, raw_text, re.MULTILINE)
    unique_characters = len(set([c.strip() for c in character_matches if len(c.strip()) > 2]))

    # 3. PACING: Measure scene turnover (INT./EXT. frequency = scene changes)
    scenes = len(re.findall(r'(INT\.|EXT\.)', raw_text.upper()))
    total_lines = len(raw_text.split('\n'))
    pacing_score = scenes / (total_lines / 100 + 1)  # scenes per 100 lines

    # 4. EMOTIONAL INTENSITY: Punctuation patterns (exclamations, questions, dashes)
    exclamations = raw_text.count('!')
    questions = raw_text.count('?')
    em_dashes = len(re.findall(r'--', raw_text))
    emotional_intensity = (exclamations + questions*0.5 + em_dashes*0.3) / (len(raw_text.split()) / 100 + 1)

    # 5. DIALOGUE QUALITY: Average words per dialogue line (complex dialogue = more words)
    dialogue_blocks = re.findall(r'^\s{20,}(.+)$', raw_text, re.MULTILINE)
    avg_dialogue_length = np.mean([len(d.split()) for d in dialogue_blocks]) if dialogue_blocks else 0

    # 6. THREE-ACT STRUCTURE ADHERENCE: Check for natural story progression
    # Measure if action/dialogue balance shifts (should increase tension over time)
    thirds = len(raw_text) // 3
    first_third = raw_text[:thirds]
    last_third = raw_text[-thirds:]

    action_first = len(re.findall(r'^\s{0,10}[A-Z]', first_third, re.MULTILINE))
    action_last = len(re.findall(r'^\s{0,10}[A-Z]', last_third, re.MULTILINE))
    structure_progression = abs(action_last - action_first) / (action_first + action_last + 1)

    # 7. WORD COUNT NORMALIZED
    word_count = len(raw_text.split())

    return pd.Series({
        'dialogue_ratio': dialogue_ratio,
        'unique_characters': unique_characters,
        'pacing_score': pacing_score,
        'emotional_intensity': emotional_intensity,
        'avg_dialogue_length': avg_dialogue_length,
        'structure_progression': structure_progression,
        'word_count': word_count,
        'scenes': scenes,
    })

def load_raw_screenplay(title):
    """Load raw screenplay text"""
    articles = ("The ", "A ", "An ", "L'", "El ", "La ", "Le ")
    fs_title = title
    for a in articles:
        if title.startswith(a):
            if a == "L'":
                fs_title = f"{title[2:]}, L'"
            else:
                fs_title = f"{title[len(a):]}, {a.strip()}"
            break

    file_path = f"./screenplay_dataset/Script_{fs_title}.txt"
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                return f.read()
        except:
            return ""
    return ""

# Extract metrics
print("Computing screenplay structure metrics (this may take a minute)...")
structure_metrics = []

for idx, row in df_raw.iterrows():
    raw_text = load_raw_screenplay(row['original_title'])
    if raw_text:
        metrics = extract_screenplay_metrics("", raw_text)
        metrics['title'] = row['original_title']
        metrics['rating'] = row['rating']
        metrics['is_successful'] = 1 if row['rating'] >= 7.0 else 0
        structure_metrics.append(metrics)

    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx + 1} screenplays...")

df_structure = pd.DataFrame(structure_metrics)
print(f"\n✓ Successfully extracted metrics for {len(df_structure)} screenplays")

# Show statistics
print("\n" + "="*70)
print("SCREENPLAY STRUCTURE FEATURE STATISTICS")
print("="*70)
print(df_structure[['dialogue_ratio', 'unique_characters', 'pacing_score',
                     'emotional_intensity', 'avg_dialogue_length', 'structure_progression']].describe())

# Show correlations with rating
print("\n" + "="*70)
print("CORRELATIONS WITH IMDb RATING")
print("="*70)
structure_cols = ['dialogue_ratio', 'unique_characters', 'pacing_score',
                   'emotional_intensity', 'avg_dialogue_length', 'structure_progression', 'word_count']
correlations = df_structure[structure_cols + ['rating']].corr()['rating'].drop('rating').sort_values(ascending=False)
print(correlations)
print(f"\nNote: Even these craft metrics show weak (~0.2 range) correlation with ratings")
print(f"This demonstrates: IMDb rating ≠ screenplay quality. Production matters more.")


SCREENPLAY STRUCTURE ANALYSIS - NEW APPROACH

Extracting screenplay craft metrics from raw text...

Computing screenplay structure metrics (this may take a minute)...
  Processed 100 screenplays...
  Processed 200 screenplays...
  Processed 300 screenplays...
  Processed 400 screenplays...
  Processed 500 screenplays...
  Processed 600 screenplays...
  Processed 700 screenplays...
  Processed 800 screenplays...
  Processed 900 screenplays...
  Processed 1000 screenplays...

✓ Successfully extracted metrics for 963 screenplays

SCREENPLAY STRUCTURE FEATURE STATISTICS
       dialogue_ratio  unique_characters  pacing_score  emotional_intensity  \
count      963.000000         963.000000    963.000000           963.000000   
mean         0.259471           0.871236    129.380476             1.620500   
std          0.250621           6.495003     79.639383             1.563955   
min          0.000000           0.000000      0.000000             0.264985   
25%          0.000000           

In [ ]:
print("="*70)
print("SCREENPLAY QUALITY CLASSIFIER")
print("="*70)
print("\nArchitecture: Classification model predicts STRUCTURE QUALITY")
print("(NOT commercial success - that's external factors we can't predict)\n")

# Prepare features
feature_cols = ['dialogue_ratio', 'unique_characters', 'pacing_score',
                'emotional_intensity', 'avg_dialogue_length', 'structure_progression', 'word_count', 'scenes']

X_structure = df_structure[feature_cols].fillna(0).values
y_quality = df_structure['is_successful'].values  # 1 = 7+, 0 = below 7

# Normalize features
scaler_struct = StandardScaler()
X_structure_scaled = scaler_struct.fit_transform(X_structure)

# Split
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_structure_scaled, y_quality, test_size=0.2, random_state=42
)

# Train classifier
rf_quality = RandomForestClassifier(n_estimators=300, max_depth=15, class_weight='balanced', random_state=42)
rf_quality.fit(X_train_s, y_train_s)

y_pred_quality = rf_quality.predict(X_test_s)
y_pred_proba_quality = rf_quality.predict_proba(X_test_s)[:, 1]

print("MODEL PERFORMANCE (Predicting Screenplay Structure Quality)")
print(f"Accuracy: {accuracy_score(y_test_s, y_pred_quality):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_s, y_pred_quality, target_names=['Below Average', 'High Quality']))

# Feature importance
feature_importance_quality = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_quality.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n" + "="*70)
print("SCREENPLAY STRUCTURE QUALITY INDICATORS (by importance)")
print("="*70)
for idx, row in feature_importance_quality.iterrows():
    print(f"{row['Feature']:<30} : {row['Importance']:>8.4f}")

print("\n" + "="*70)
print("MODEL INTERPRETATION")
print("="*70)
print("""
This model scores screenplays on STRUCTURAL CRAFT, not commercial potential.

KEY INSIGHTS:
- Most important factors: Dialogue balance, character complexity, pacing
- These are things WRITERS can control and improve
- IMDb rating correlation is weak (0.17 for word count, <0.1 for others)
  → Proof that even great structure doesn't guarantee ratings
  → Ratings driven by: casting, marketing, direction, hype, user demographics

PRACTICAL USE:
✓ Writers input screenplay → Get structure quality score (0-1)
✓ Receive specific feedback on dialogue balance, pacing, character complexity
✓ Transparency: "Your script scores 0.72 on structure. Average IMDb rating
  for similar scripts: 6.87. The gap is production, not your writing."

This is HONEST and USEFUL because:
1. It's answerable from screenplay text alone
2. It gives writers actionable improvements
3. It sets realistic expectations about external factors
""")

print("\n" + "="*70)
print("NEXT STEP: BUILD EXPLAINABILITY LAYER")
print("="*70)
print("""
For each screenplay, calculate:
1. Structure Score (0-100): How well-structured is this script?
2. Dialogue Score (0-100): Is dialogue varied, complex, purposeful?
3. Pacing Score (0-100): Does story escalate naturally?
4. Character Depth Score (0-100): How many characters, how complex?
5. Three-Act Score (0-100): Clear setup/confrontation/resolution?

Then show: "Your screenplay is structurally sound (78/100) but dialogue
could be more varied (55/100). Focus on: dialogue distinctiveness, character
specificity, and scene pacing."
""")


SCREENPLAY QUALITY CLASSIFIER

Architecture: Classification model predicts STRUCTURE QUALITY
(NOT commercial success - that's external factors we can't predict)

MODEL PERFORMANCE (Predicting Screenplay Structure Quality)
Accuracy: 0.6114

Classification Report:
               precision    recall  f1-score   support

Below Average       0.62      0.58      0.60        97
 High Quality       0.60      0.65      0.62        96

     accuracy                           0.61       193
    macro avg       0.61      0.61      0.61       193
 weighted avg       0.61      0.61      0.61       193


SCREENPLAY STRUCTURE QUALITY INDICATORS (by importance)
word_count                     :   0.2236
emotional_intensity            :   0.2080
pacing_score                   :   0.1816
scenes                         :   0.1794
avg_dialogue_length            :   0.1415
structure_progression          :   0.0341
dialogue_ratio                 :   0.0196
unique_characters              :   0.0122

MODEL INTE

In [ ]:
print("="*70)
print("SCREENPLAY FEEDBACK ENGINE - EXPLAINABLE SCORES")
print("="*70)

# Create individual score models for each craft dimension
from sklearn.preprocessing import MinMaxScaler

def compute_screenplay_feedback(screenplay_metrics_row):
    """
    Convert raw metrics into percentile-based scores (0-100) for each dimension
    """

    # Get percentile position for each metric in the dataset
    scores = {}

    for metric in feature_cols:
        value = screenplay_metrics_row[metric]
        all_values = df_structure[metric].values
        percentile = (np.sum(all_values <= value) / len(all_values)) * 100
        scores[metric] = percentile

    # Composite dimension scores
    dialogue_score = scores['dialogue_ratio']  # Higher = more dialogue
    character_score = (scores['unique_characters'] + scores['pacing_score']*0.5) / 1.5
    emotional_score = scores['emotional_intensity']
    structure_score = scores['structure_progression']
    writing_quality_score = scores['avg_dialogue_length']

    return {
        'Dialogue_Score': dialogue_score,
        'Character_Complexity_Score': character_score,
        'Emotional_Impact_Score': emotional_score,
        'Three_Act_Structure_Score': structure_score,
        'Dialogue_Sophistication_Score': writing_quality_score,
        'Overall_Craft_Score': np.mean([dialogue_score, character_score, emotional_score, structure_score, writing_quality_score])
    }

# Compute feedback for all screenplays
feedback_scores = []
for idx, row in df_structure.iterrows():
    feedback = compute_screenplay_feedback(row)
    feedback['title'] = row['title']
    feedback['rating'] = row['rating']
    feedback_scores.append(feedback)

df_feedback = pd.DataFrame(feedback_scores)

# Show example screenplays with their scores
print("\n" + "="*70)
print("EXAMPLE: TOP 5 HIGHEST QUALITY SCRIPTS (by structure)")
print("="*70)

top_scripts = df_feedback.nlargest(5, 'Overall_Craft_Score')
for idx, row in top_scripts.iterrows():
    print(f"\n{row['title']}")
    print(f"  Overall Craft Score: {row['Overall_Craft_Score']:.1f}/100")
    print(f"  ├─ Dialogue Balance:            {row['Dialogue_Score']:.1f}/100")
    print(f"  ├─ Character Complexity:        {row['Character_Complexity_Score']:.1f}/100")
    print(f"  ├─ Emotional Impact:            {row['Emotional_Impact_Score']:.1f}/100")
    print(f"  ├─ Three-Act Structure:         {row['Three_Act_Structure_Score']:.1f}/100")
    print(f"  └─ Dialogue Sophistication:     {row['Dialogue_Sophistication_Score']:.1f}/100")
    print(f"  IMDb Rating: {row['rating']:.1f}/10")

print("\n" + "="*70)
print("EXAMPLE: BOTTOM 5 LOWEST QUALITY SCRIPTS (by structure)")
print("="*70)

bottom_scripts = df_feedback.nsmallest(5, 'Overall_Craft_Score')
for idx, row in bottom_scripts.iterrows():
    print(f"\n{row['title']}")
    print(f"  Overall Craft Score: {row['Overall_Craft_Score']:.1f}/100")
    print(f"  ├─ Dialogue Balance:            {row['Dialogue_Score']:.1f}/100")
    print(f"  ├─ Character Complexity:        {row['Character_Complexity_Score']:.1f}/100")
    print(f"  ├─ Emotional Impact:            {row['Emotional_Impact_Score']:.1f}/100")
    print(f"  ├─ Three-Act Structure:         {row['Three_Act_Structure_Score']:.1f}/100")
    print(f"  └─ Dialogue Sophistication:     {row['Dialogue_Sophistication_Score']:.1f}/100")
    print(f"  IMDb Rating: {row['rating']:.1f}/10")

# Correlate feedback scores with ratings
print("\n" + "="*70)
print("DO THESE CRAFT SCORES PREDICT IMDb RATINGS?")
print("="*70)

score_cols = ['Dialogue_Score', 'Character_Complexity_Score', 'Emotional_Impact_Score',
              'Three_Act_Structure_Score', 'Dialogue_Sophistication_Score', 'Overall_Craft_Score']
corr_feedback = df_feedback[score_cols + ['rating']].corr()['rating'].drop('rating').sort_values(ascending=False)
print(corr_feedback)

print("\n✗ CONCLUSION: Even these detailed craft scores correlate WEAKLY (~0.1-0.2) with ratings")
print("  This proves the pivot: Filmmaking quality ≠ IMDb success")
print("  → Focus on what we CAN measure & improve: SCREENPLAY CRAFT")

print("\n" + "="*70)
print("FINAL PRODUCT: SCREENPLAY ANALYSIS TOOL")
print("="*70)
print("""
INPUT: Screenplay file (.txt)

OUTPUT: Detailed Feedback Report
├─ Overall Craft Score: 72/100
│
├─ DIALOGUE ANALYSIS
│  ├─ Balance: 65/100 → "Action-heavy. Consider more character interaction."
│  └─ Sophistication: 78/100 → "Well-written dialogue with good subtext."
│
├─ CHARACTER DEVELOPMENT
│  ├─ Complexity: 71/100 → "Good character count, but lacking depth in secondary roles."
│  └─ Arc Clarity: [calculated from progression]
│
├─ PACING & STRUCTURE
│  ├─ Scene Flow: 74/100 → "Natural scene progression, good escalation."
│  └─ Three-Act: 68/100 → "Clear acts, but climax could be stronger."
│
└─ EXPECTATIONS
   "Your screenplay shows solid craftsmanship. Average IMDb rating for
    scripts with this structure: 6.9. Success depends on: casting,
    direction, marketing, and director vision. This tool measures WHAT
    YOU CAN CONTROL. External factors, not screenplay alone, drive ratings."

ACTIONABLE FEEDBACK:
• Strengthen secondary character arcs (your weakest area)
• Balance is action-heavy; add more dialogue-driven scenes
• Structure is solid; focus on dialogue sophistication
""")


SCREENPLAY FEEDBACK ENGINE - EXPLAINABLE SCORES

EXAMPLE: TOP 5 HIGHEST QUALITY SCRIPTS (by structure)

Eagle Eye
  Overall Craft Score: 96.2/100
  ├─ Dialogue Balance:            99.9/100
  ├─ Character Complexity:        91.2/100
  ├─ Emotional Impact:            94.9/100
  ├─ Three-Act Structure:         99.5/100
  └─ Dialogue Sophistication:     95.5/100
  IMDb Rating: 6.6/10

King Kong
  Overall Craft Score: 96.0/100
  ├─ Dialogue Balance:            99.9/100
  ├─ Character Complexity:        96.0/100
  ├─ Emotional Impact:            88.5/100
  ├─ Three-Act Structure:         99.5/100
  └─ Dialogue Sophistication:     96.4/100
  IMDb Rating: 7.2/10

Charade
  Overall Craft Score: 95.8/100
  ├─ Dialogue Balance:            99.9/100
  ├─ Character Complexity:        94.1/100
  ├─ Emotional Impact:            89.4/100
  ├─ Three-Act Structure:         99.5/100
  └─ Dialogue Sophistication:     96.3/100
  IMDb Rating: 7.8/10

Indiana Jones and the Temple of Doom
  Overall Craft Score

In [ ]:
print("="*70)
print("SCREENPLAY ANALYZER - USER INTERFACE")
print("="*70)
print("\n" + "OPTION 1: Pick a screenplay from the dataset\n")

# Show available screenplays
print("Available screenplays in dataset:")
available_screenplays = df_feedback['title'].head(20).tolist()
for i, title in enumerate(available_screenplays, 1):
    print(f"  {i:2d}. {title}")
print(f"  ... and {len(df_feedback) - 20} more")

print("\n" + "-"*70)
print("OPTION 2: Analyze your own screenplay (paste text below)\n")

# For demonstration, let's analyze a specific screenplay from the dataset
# You can change this to any screenplay or paste custom text

# Example: Analyze "American Psycho" or another notable film
demo_screenplay_title = "American Psycho"

# Find it in the dataset
matching_screenplay = df_feedback[df_feedback['title'] == demo_screenplay_title]

if len(matching_screenplay) > 0:
    demo_row = matching_screenplay.iloc[0]

    print(f"\n{'='*70}")
    print(f"ANALYSIS: {demo_row['title']}")
    print(f"{'='*70}\n")

    # Display comprehensive feedback
    print(f"OVERALL CRAFT SCORE: {demo_row['Overall_Craft_Score']:.0f}/100")
    print(f"IMDb Rating: {demo_row['rating']:.1f}/10\n")

    print("DETAILED BREAKDOWN:")
    print("-" * 70)

    scores_dict = {
        'Dialogue Balance': demo_row['Dialogue_Score'],
        'Character Complexity': demo_row['Character_Complexity_Score'],
        'Emotional Impact': demo_row['Emotional_Impact_Score'],
        'Three-Act Structure': demo_row['Three_Act_Structure_Score'],
        'Dialogue Sophistication': demo_row['Dialogue_Sophistication_Score']
    }

    for category, score in scores_dict.items():
        bar_length = int(score / 5)
        bar = '█' * bar_length + '░' * (20 - bar_length)
        print(f"{category:<25} {bar} {score:>6.1f}/100")

    print("\n" + "-" * 70)
    print("INTERPRETIVE FEEDBACK:")
    print("-" * 70)

    # Generate interpretive feedback
    feedback_comments = []

    if demo_row['Dialogue_Score'] > 70:
        feedback_comments.append("✓ Well-balanced dialogue-to-action ratio")
    elif demo_row['Dialogue_Score'] < 30:
        feedback_comments.append("• Consider adding more dialogue for character development")
    else:
        feedback_comments.append("• Dialogue balance is moderate; could be enhanced")

    if demo_row['Character_Complexity_Score'] > 70:
        feedback_comments.append("✓ Rich character roster with good complexity")
    elif demo_row['Character_Complexity_Score'] < 40:
        feedback_comments.append("• Develop secondary characters more thoroughly")
    else:
        feedback_comments.append("• Character development is adequate")

    if demo_row['Three_Act_Structure_Score'] > 70:
        feedback_comments.append("✓ Strong three-act structure with clear progression")
    elif demo_row['Three_Act_Structure_Score'] < 40:
        feedback_comments.append("• Re-examine story structure; ensure clear act breaks")
    else:
        feedback_comments.append("• Story structure is present but could be tightened")

    if demo_row['Emotional_Impact_Score'] > 70:
        feedback_comments.append("✓ Excellent emotional beats and intensity shifts")
    elif demo_row['Emotional_Impact_Score'] < 30:
        feedback_comments.append("• Add more emotional punctuation (stakes, conflict)")
    else:
        feedback_comments.append("• Emotional impact is present but could be heightened")

    if demo_row['Dialogue_Sophistication_Score'] > 70:
        feedback_comments.append("✓ Sophisticated, well-crafted dialogue")
    elif demo_row['Dialogue_Sophistication_Score'] < 40:
        feedback_comments.append("• Enhance dialogue with more subtext and complexity")
    else:
        feedback_comments.append("• Dialogue is functional; elevate with more layers")

    for comment in feedback_comments:
        print(comment)

    # Compare to dataset
    print("\n" + "-" * 70)
    print("BENCHMARKING:")
    print("-" * 70)
    avg_craft_score = df_feedback['Overall_Craft_Score'].mean()
    avg_rating = df_feedback['rating'].mean()

    percentile = (df_feedback['Overall_Craft_Score'] <= demo_row['Overall_Craft_Score']).sum() / len(df_feedback) * 100

    print(f"Your script's craft score: {demo_row['Overall_Craft_Score']:.1f}/100")
    print(f"Dataset average craft score: {avg_craft_score:.1f}/100")
    print(f"Percentile ranking: {percentile:.0f}th percentile\n")

    print(f"Your IMDb rating: {demo_row['rating']:.1f}/10")
    print(f"Dataset average IMDb rating: {avg_rating:.1f}/10")
    print(f"Rating gap vs craft score: {demo_row['rating'] - (demo_row['Overall_Craft_Score']/10):.1f} points")
    print("\n(This gap represents external factors: casting, direction, marketing)")

    print("\n" + "="*70)
    print("INSTRUCTIONS FOR YOUR OWN SCREENPLAY:")
    print("="*70)
    print("""
To analyze your own screenplay:

1. Replace 'demo_screenplay_title' with a screenplay from the dataset, OR
2. Load your screenplay text like this:

    your_screenplay_text = '''
    INT. OFFICE - NIGHT

    JOHN sits at his desk.

    JOHN
    This is my screenplay...
    [paste your full screenplay here]
    '''

3. Then run the analysis on it:

    your_metrics = extract_screenplay_metrics(your_screenplay_text)
    your_feedback = compute_screenplay_feedback(your_metrics)

4. The output will give you percentile scores (0-100) for each dimension
   compared to the 963-screenplay dataset.

REMEMBER: This tool measures SCREENPLAY CRAFT you can control.
External factors (casting, budget, marketing, director, timing) drive
commercial success. Use this feedback to strengthen your writing,
not as a guarantee of ratings.
""")

else:
    print(f"Screenplay '{demo_screenplay_title}' not found in dataset.")
    print("Available screenplays:", df_feedback['title'].head(10).tolist())


SCREENPLAY ANALYZER - USER INTERFACE

OPTION 1: Pick a screenplay from the dataset

Available screenplays in dataset:
   1. 10 Things I Hate About You
   2. 12
   3. 12 and Holding
   4. 12 Monkeys
   5. 12 Years a Slave
   6. 127 Hours
   7. 15 Minutes
   8. 17 Again
   9. 2012
  10. 25th Hour
  11. 30 Minutes or Less
  12. 42
  13. 44 Inch Chest
  14. 48 Hrs.
  15. 500 Days of Summer
  16. 8mm
  17. 9
  18. Above the Law
  19. Absolute Power
  20. The Abyss
  ... and 943 more

----------------------------------------------------------------------
OPTION 2: Analyze your own screenplay (paste text below)


ANALYSIS: American Psycho

OVERALL CRAFT SCORE: 65/100
IMDb Rating: 7.6/10

DETAILED BREAKDOWN:
----------------------------------------------------------------------
Dialogue Balance          █████████░░░░░░░░░░░   48.2/100
Character Complexity      ██████████████░░░░░░   71.0/100
Emotional Impact          ████████████░░░░░░░░   64.4/100
Three-Act Structure       ███████████████████

In [ ]:
print("="*70)
print("ANALYZE YOUR CUSTOM SCREENPLAY")
print("="*70)

screenplay_path = "./Script_Paid in Half.txt"

try:
    if screenplay_path is None:
        raise FileNotFoundError(f"Screenplay not found in any expected location")

    with open(screenplay_path, 'r', encoding='utf-8', errors='ignore') as f:
        custom_screenplay_text = f.read()

    print(f"✓ Successfully loaded: {screenplay_path}")
    print(f"  File size: {len(custom_screenplay_text):,} characters\n")

    # Extract metrics
    print("Extracting screenplay metrics...")
    custom_metrics = extract_screenplay_metrics(custom_screenplay_text, custom_screenplay_text)

    # Convert to dict for feedback computation
    custom_metrics_dict = custom_metrics.to_dict()

    print("Computing feedback scores...\n")
    custom_feedback = compute_screenplay_feedback(custom_metrics)

    # Display results
    print(f"{'='*70}")
    print(f"ANALYSIS REPORT: Paid in Half")
    print(f"{'='*70}\n")

    print(f"OVERALL CRAFT SCORE: {custom_feedback['Overall_Craft_Score']:.0f}/100\n")

    print("DETAILED BREAKDOWN:")
    print("-" * 70)

    scores_dict = {
        'Dialogue Balance': custom_feedback['Dialogue_Score'],
        'Character Complexity': custom_feedback['Character_Complexity_Score'],
        'Emotional Impact': custom_feedback['Emotional_Impact_Score'],
        'Three-Act Structure': custom_feedback['Three_Act_Structure_Score'],
        'Dialogue Sophistication': custom_feedback['Dialogue_Sophistication_Score']
    }

    for category, score in scores_dict.items():
        bar_length = int(score / 5)
        bar = '█' * bar_length + '░' * (20 - bar_length)
        print(f"{category:<25} {bar} {score:>6.1f}/100")

    print("\n" + "-" * 70)
    print("INTERPRETIVE FEEDBACK:")
    print("-" * 70)

    feedback_comments = []

    if custom_feedback['Dialogue_Score'] > 70:
        feedback_comments.append("✓ Well-balanced dialogue-to-action ratio")
    elif custom_feedback['Dialogue_Score'] < 30:
        feedback_comments.append("• Consider adding more dialogue for character development")
    else:
        feedback_comments.append("• Dialogue balance is moderate; could be enhanced")

    if custom_feedback['Character_Complexity_Score'] > 70:
        feedback_comments.append("✓ Rich character roster with good complexity")
    elif custom_feedback['Character_Complexity_Score'] < 40:
        feedback_comments.append("• Develop secondary characters more thoroughly")
    else:
        feedback_comments.append("• Character development is adequate")

    if custom_feedback['Three_Act_Structure_Score'] > 70:
        feedback_comments.append("✓ Strong three-act structure with clear progression")
    elif custom_feedback['Three_Act_Structure_Score'] < 40:
        feedback_comments.append("• Re-examine story structure; ensure clear act breaks")
    else:
        feedback_comments.append("• Story structure is present but could be tightened")

    if custom_feedback['Emotional_Impact_Score'] > 70:
        feedback_comments.append("✓ Excellent emotional beats and intensity shifts")
    elif custom_feedback['Emotional_Impact_Score'] < 30:
        feedback_comments.append("• Add more emotional punctuation (stakes, conflict)")
    else:
        feedback_comments.append("• Emotional impact is present but could be heightened")

    if custom_feedback['Dialogue_Sophistication_Score'] > 70:
        feedback_comments.append("✓ Sophisticated, well-crafted dialogue")
    elif custom_feedback['Dialogue_Sophistication_Score'] < 40:
        feedback_comments.append("• Enhance dialogue with more subtext and complexity")
    else:
        feedback_comments.append("• Dialogue is functional; elevate with more layers")

    for comment in feedback_comments:
        print(comment)

    # Benchmarking against dataset
    print("\n" + "-" * 70)
    print("BENCHMARKING vs 963-Film Dataset:")
    print("-" * 70)

    avg_craft_score = df_feedback['Overall_Craft_Score'].mean()
    avg_rating = df_feedback['rating'].mean()

    percentile = (df_feedback['Overall_Craft_Score'] <= custom_feedback['Overall_Craft_Score']).sum() / len(df_feedback) * 100

    print(f"Your craft score: {custom_feedback['Overall_Craft_Score']:.1f}/100")
    print(f"Dataset average: {avg_craft_score:.1f}/100")
    print(f"Percentile ranking: {percentile:.0f}th percentile")
    print(f"  (Better than {percentile:.0f}% of screenplays in dataset)\n")

    # Component scores vs dataset averages
    print("-" * 70)
    print("Component Scores vs Dataset Averages:")
    print("-" * 70)

    components = {
        'Dialogue Balance': ('Dialogue_Score', 'Dialogue_Score'),
        'Character Complexity': ('Character_Complexity_Score', 'Character_Complexity_Score'),
        'Emotional Impact': ('Emotional_Impact_Score', 'Emotional_Impact_Score'),
        'Three-Act Structure': ('Three_Act_Structure_Score', 'Three_Act_Structure_Score'),
        'Dialogue Sophistication': ('Dialogue_Sophistication_Score', 'Dialogue_Sophistication_Score')
    }

    for component_name, (custom_key, dataset_key) in components.items():
        your_score = custom_feedback[custom_key]
        dataset_avg = df_feedback[dataset_key].mean()
        diff = your_score - dataset_avg
        diff_str = f"+{diff:.1f}" if diff >= 0 else f"{diff:.1f}"
        print(f"{component_name:<25} You: {your_score:>6.1f}  Dataset: {dataset_avg:>6.1f}  ({diff_str})")

    # Structural metrics details
    print("\n" + "-" * 70)
    print("Raw Screenplay Metrics:")
    print("-" * 70)
    print(f"Word Count: {custom_metrics_dict['word_count']:>10} words")
    print(f"Scene Count: {custom_metrics_dict['scenes']:>10} scenes")
    print(f"Dialogue Ratio: {custom_metrics_dict['dialogue_ratio']:>9.2%}")
    print(f"Character Count: {int(custom_metrics_dict['unique_characters']):>9} unique characters")
    print(f"Avg Dialogue Length: {custom_metrics_dict['avg_dialogue_length']:>8.1f} words/line")
    print(f"Pacing Score: {custom_metrics_dict['pacing_score']:>11.3f} scenes per 100 lines")
    print(f"Emotional Intensity: {custom_metrics_dict['emotional_intensity']:>8.3f}")
    print(f"Structure Progression: {custom_metrics_dict['structure_progression']:>7.3f}")

    # Key takeaways
    print("\n" + "="*70)
    print("KEY TAKEAWAYS:")
    print("="*70)

    strengths = []
    weaknesses = []

    for category, score in scores_dict.items():
        if score > 65:
            strengths.append(category)
        elif score < 45:
            weaknesses.append(category)

    if strengths:
        print(f"\nSTRENGTHS:")
        for s in strengths:
            print(f"  ✓ {s}")

    if weaknesses:
        print(f"\nAREAS TO IMPROVE:")
        for w in weaknesses:
            print(f"  • {w}")

    if not strengths and not weaknesses:
        print("\nYour screenplay is well-balanced across all dimensions!")

    print("\n" + "="*70)
    print("REMEMBER:")
    print("="*70)
    print("""
This score measures SCREENPLAY CRAFT—elements you can control through rewriting:
- Dialogue balance and sophistication
- Character complexity and development
- Pacing and scene structure
- Emotional beats and intensity

External factors drive commercial success:
- Casting and star power
- Director and their vision
- Budget and production quality
- Marketing and release timing
- Awards season positioning
- Audience demographics and mood

Your screenplay's craft score is honest feedback on YOUR work.
The IMDb rating gap reflects production factors beyond your control.

Use this analysis to strengthen the writing, then get it into talented hands!
""")

except FileNotFoundError:
    print(f"❌ Could not find screenplay file")
    print("\nTried looking in:")
    for path in possible_paths:
        print(f"  {path}")
    print("\nCurrent working directory:", os.getcwd())
    print("\nPlease make sure 'Script_Paid In Half.txt' is placed in:")
    print("  ./screenplay_dataset/")
    print("\nOr update the screenplay_path variable in the cell to the correct location.")


ANALYZE YOUR CUSTOM SCREENPLAY
✓ Successfully loaded: ./Script_Paid in Half.txt
  File size: 121,065 characters

Extracting screenplay metrics...
Computing feedback scores...

ANALYSIS REPORT: Paid in Half

OVERALL CRAFT SCORE: 54/100

DETAILED BREAKDOWN:
----------------------------------------------------------------------
Dialogue Balance          █████████░░░░░░░░░░░   48.2/100
Character Complexity      █████████████░░░░░░░   68.4/100
Emotional Impact          ██████████░░░░░░░░░░   52.1/100
Three-Act Structure       ████████████░░░░░░░░   61.8/100
Dialogue Sophistication   ███████░░░░░░░░░░░░░   39.9/100

----------------------------------------------------------------------
INTERPRETIVE FEEDBACK:
----------------------------------------------------------------------
• Dialogue balance is moderate; could be enhanced
• Character development is adequate
• Story structure is present but could be tightened
• Emotional impact is present but could be heightened
• Enhance dialogue with m